# MongoDB Advanced: How to Inspect a Replica Set and Secure Access

So far you have read, written, aggregated, designed schemas, and run transactions against a single logical database. This lab is the step to production: how does the data **stay available** if a machine dies, and how is **access restricted** to the right people? You will inspect a real running Atlas cluster and verify the features that keep a database alive and secure.

You will learn how to:
1. Connect to a MongoDB Atlas cluster
2. See that you are talking to a live 3-node replica set
3. Spot the primary and its secondaries, and learn what the oplog is for
4. Read from a secondary with a read preference
5. Verify the connection is encrypted with TLS
6. Work with role-based access control (RBAC) using built-in roles
7. Understand sharding conceptually
8. Generate a summary report and leave the shared cluster clean

In [1]:
!pip install -qU "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


This installs the MongoDB Python driver (`pymongo`) with `srv` and `tls` extras, `python-dotenv` for loading credentials, and `certifi` for CA certificates.

## A Note About the Free Tier: We Inspect, We Do Not Provision

The free M0 cluster is already a live 3-node replica set, TLS is already enforced on every connection, and built-in RBAC roles work. This lab **inspects and verifies** those features; it does not provision them. Adding nodes or shards is a paid, DevOps concern beyond this module — the full picture is in the companion `.md`.

### Step 1 — Connect to MongoDB

In [2]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file one directory up
load_dotenv("../../.env")
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collections used in earlier labs
db = client["school_db"]
students = db["students"]
courses = db["courses"]

print("Connected to MongoDB Atlas")

Connected to MongoDB Atlas


Same connection pattern as since Lab 1, reading the shared `.env`. We pass `tlsCAFile` — our first hint that every Atlas connection is encrypted (confirmed in Step 5).

### Step 2 — See That You're Talking to a Replica Set

In [3]:
# The "hello" command reports back the node's view of the cluster
hello = db.command("hello")

print("setName:", hello.get("setName"))
print("me:", hello.get("me"))
print("isWritablePrimary:", hello.get("isWritablePrimary"))
print("\nhosts in this replica set:")
for host in hello.get("hosts", []):
    print("  ", host)
print("\ncurrent primary:")
print("  ", hello.get("primary"))

setName: atlas-jooykf-shard-0
me: ac-mtwetuu-shard-00-01.rfzvgwe.mongodb.net:27017
isWritablePrimary: True

hosts in this replica set:
   ac-mtwetuu-shard-00-00.rfzvgwe.mongodb.net:27017
   ac-mtwetuu-shard-00-01.rfzvgwe.mongodb.net:27017
   ac-mtwetuu-shard-00-02.rfzvgwe.mongodb.net:27017

current primary:
   ac-mtwetuu-shard-00-01.rfzvgwe.mongodb.net:27017


The `hello` command reports the node's view of the cluster: `setName` proves this is a **replica set**, `hosts` lists all 3 nodes, and `primary` names the current leader. Writes go only to the primary. Full explanation in the `.md`.

### Step 3 — Primary vs Secondaries, and the Oplog Idea

In [4]:
# Label each node in the replica set as primary or secondary
node_roles = []
for addr in sorted(client.nodes):
    node = f"{addr[0]}:{addr[1]}"
    if node == hello["primary"]:
        node_roles.append((node, "PRIMARY"))
    else:
        node_roles.append((node, "SECONDARY"))

print("Replica set members:")
for node, role in node_roles:
    print(f"  {node}  ->  {role}")

Replica set members:
  ac-mtwetuu-shard-00-00.rfzvgwe.mongodb.net:27017  ->  SECONDARY
  ac-mtwetuu-shard-00-01.rfzvgwe.mongodb.net:27017  ->  PRIMARY
  ac-mtwetuu-shard-00-02.rfzvgwe.mongodb.net:27017  ->  SECONDARY


We label each node in `client.nodes` as primary or secondary. Writes go to the primary and replicate to secondaries via the **oplog**; if the primary dies, a secondary is elected (automatic failover). Full explanation in the `.md`.

### Step 4 — Read From a Secondary

In [5]:
from pymongo import ReadPreference

# Seed one clearly-tagged document so we have data to read (cleaned up in Step 8)
students.insert_one({
    "student_id": "LAB6TAG001",
    "name": "Replica Read Test",
    "lab6_marker": True
})

# A database handle that PREFERS secondaries for reads
secondary_db = client.get_database(
    "school_db", read_preference=ReadPreference.SECONDARY_PREFERRED
)

# Read the tagged doc through the primary ...
primary_view = db["students"].find_one({"student_id": "LAB6TAG001"})
# ... and through a secondary-preferred handle
secondary_view = secondary_db["students"].find_one({"student_id": "LAB6TAG001"})

print("Read via primary:      ", primary_view["name"])
print("Read via secondary:    ", secondary_view["name"])

Read via primary:       Replica Read Test
Read via secondary:     Replica Read Test


A **read preference** lets secondaries serve reads, spreading load across the set. `SECONDARY_PREFERRED` uses a secondary when available; the trade-off is possible slight staleness (eventual consistency). Full explanation in the `.md`.

### Step 5 — Verify the Connection Is Encrypted (TLS)

In [6]:
import os

# The mongodb+srv scheme REQUIRES TLS for every connection by driver spec
scheme = uri.split("://", 1)[0]
print("URI scheme:", scheme)
print("TLS is mandatory for mongodb+srv:", scheme == "mongodb+srv")

# We supplied a trusted CA bundle via tlsCAFile
ca_path = certifi.where()
print("CA bundle file exists:", os.path.exists(ca_path))
print("CA bundle path:", ca_path)

URI scheme: mongodb+srv
TLS is mandatory for mongodb+srv: True
CA bundle file exists: True
CA bundle path: C:\Users\risha\AppData\Local\Programs\Python\Python312\Lib\site-packages\certifi\cacert.pem


**TLS** encrypts every connection so data in transit is protected. We verify it: the `mongodb+srv` scheme requires TLS, and `tlsCAFile` supplies a trusted CA bundle. Atlas enforces TLS on every connection. Full explanation in the `.md`.

### Step 6 — Role-Based Access Control (RBAC)

In [7]:
# Ask the server who we are and what we are allowed to do
status = db.command("connectionStatus")

print("Authenticated user:")
for u in status["authInfo"]["authenticatedUsers"]:
    print("  ", u.get("user"), "on", u.get("db"))

print("\nCurrent user's roles:")
for r in status["authInfo"]["authenticatedUserRoles"]:
    print("  -", r.get("role"), "on db", r.get("db"))

Authenticated user:
   rishabhpatidar400_db_user on admin

Current user's roles:
  - atlasAdmin on db admin


**RBAC** ties access to roles; `connectionStatus` shows the current user's roles. **Least privilege** grants only what each app needs. Creating a read-only user (say, for a dashboard) is done in the Atlas UI — numbered steps in the `.md`.

### Step 7 — Sharding (Conceptual Only)

**Sharding** spreads a collection across many servers when one is not enough; a router (`mongos`) routes by **shard key**. The free M0 tier cannot shard, so this stays conceptual — diagram and detail in the `.md`.

### Step 8 — Summary Report and Cleanup

In [8]:
# --- Clean up the document we seeded in Step 4 (tagged deletes) ---
removed = students.delete_many({"lab6_marker": True})
print(f"Removed {removed.deleted_count} lab-6 seeded document(s).")

# --- Summary of everything we verified ---
print("\n       REPLICA SET & SECURITY CHECKLIST")
print("\nReplica set name:      ", hello.get("setName"))
print(f"Member node count:     {len(hello.get('hosts', []))}")
print("Primary host:          ", hello.get("primary"))
print("Connection encrypted:   TLS enforced (mongodb+srv + tlsCAFile)")
print("Current user:          ", next((u.get('user') for u in status['authInfo']['authenticatedUsers']), 'unknown'))
print("Current user's roles:  ", ", ".join(r.get('role') for r in status['authInfo']['authenticatedUserRoles']))

Removed 1 lab-6 seeded document(s).

       REPLICA SET & SECURITY CHECKLIST

Replica set name:       atlas-jooykf-shard-0
Member node count:     3
Primary host:           ac-mtwetuu-shard-00-01.rfzvgwe.mongodb.net:27017
Connection encrypted:   TLS enforced (mongodb+srv + tlsCAFile)
Current user:           rishabhpatidar400_db_user
Current user's roles:   atlasAdmin


We verified a live replica set, secondary reads, TLS on every connection, RBAC roles, and the sharding concept — the properties behind the Lab 7 capstone's **university registration database**. The cluster was left clean; our tagged seed doc was removed in Step 8.